In [2]:
import pandas as pd
import numpy as np

In [38]:
import pyodbc


conn = pyodbc.connect(
    'DRIVER={ODBC Driver 18 for SQL Server};'
    'SERVER=phazd1856sqlserver.database.windows.net;'
    'DATABASE=devsqldatabase;'
    'UID=administratorLogin;'
    'PWD=Admin@8874;'
    'Encrypt=yes;'
    'TrustServerCertificate=no;'
    'Connection Timeout=100;'
)
cursor = conn.cursor()
print(" Connected to Azure SQL Server successfully!")


 Connected to Azure SQL Server successfully!


## provience_insert

In [3]:
prov = pd.read_csv("province_master_Thai.csv")
prov

,ID,Province Name (Eng),Province Name (Thai),isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,LOP BURI,ลพบุรี,1,NaN,NaN,NaN,NaN,7
1,2,SARABURI,สระบุรี,1,NaN,NaN,NaN,NaN,7
2,3,NAKHON SAWAN,นครสวรรค์,1,NaN,NaN,NaN,NaN,7
3,4,UTHAI THANI,อุทัยธานี,1,NaN,NaN,NaN,NaN,7
4,5,KANCHANABURI,กาญจนบุรี,1,NaN,NaN,NaN,NaN,7
...,...,...,...,...,...,...,...,...,...
72,73,TRANG,ตรัง,1,NaN,NaN,NaN,NaN,7
73,74,PHATTHALUNG,พัทลุง,1,NaN,NaN,NaN,NaN,7
74,75,PATTANI,ปัตตานี,1,NaN,NaN,NaN,NaN,7
75,76,YALA,ยะลา,1,NaN,NaN,NaN,NaN,7


In [5]:
from datetime import datetime
prov['createdon']= datetime.now()
prov

,ID,Province Name (Eng),Province Name (Thai),isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,LOP BURI,ลพบุรี,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7
1,2,SARABURI,สระบุรี,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7
2,3,NAKHON SAWAN,นครสวรรค์,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7
3,4,UTHAI THANI,อุทัยธานี,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7
4,5,KANCHANABURI,กาญจนบุรี,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7
...,...,...,...,...,...,...,...,...,...
72,73,TRANG,ตรัง,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7
73,74,PHATTHALUNG,พัทลุง,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7
74,75,PATTANI,ปัตตานี,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7
75,76,YALA,ยะลา,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7


In [6]:
prov['modifiedon'] = pd.to_datetime(prov['modifiedon'],errors = 'coerce')
prov['createdon'] = pd.to_datetime(prov['createdon'],errors = 'coerce')
prov.head(3)

,ID,Province Name (Eng),Province Name (Thai),isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,LOP BURI,ลพบุรี,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7
1,2,SARABURI,สระบุรี,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7
2,3,NAKHON SAWAN,นครสวรรค์,1,2025-09-17 10:41:59.352604,NaT,NaN,NaN,7


In [17]:
prov['createdby'] = 'Admin'

In [18]:
prov.isna().sum()

ID                       0
Province Name (Eng)      0
Province Name (Thai)     0
isActive                 0
createdon                0
modifiedon              77
createdby                0
modifiedby              77
CountryID                0
dtype: int64

In [8]:
prov.columns

Index(['ID', 'Province Name (Eng)', 'Province Name (Thai)', 'isActive',
       'createdon', 'modifiedon', 'createdby', 'modifiedby', 'CountryID'],
      dtype='object')

In [15]:
prov.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   ID                    77 non-null     int64         
 1   Province Name (Eng)   77 non-null     object        
 2   Province Name (Thai)  77 non-null     object        
 3   isActive              77 non-null     int64         
 4   createdon             77 non-null     datetime64[us]
 5   modifiedon            0 non-null      datetime64[ns]
 6   createdby             0 non-null      float64       
 7   modifiedby            0 non-null      float64       
 8   CountryID             77 non-null     int64         
dtypes: datetime64[ns](1), datetime64[us](1), float64(2), int64(3), object(2)
memory usage: 5.5+ KB


In [20]:
cursor.execute("SET IDENTITY_INSERT [tl].[PoliticalProvinceMaster] ON") 
# modifiedBy,
#  modifiedOn,
for _, row in prov.iterrows():
    cursor.execute("""
        INSERT INTO [tl].[PoliticalProvinceMaster] (
            politicalProvinceId,
            createdBy,
            
            createdOn,
           
            isActive,
            countryId,
            provinceName,
            provinceNameInNative
        ) VALUES (?, ?, ?, ?, ?, ?, ?)
    """, 
        row['ID'], 
        row['createdby'], 
#         row['modifiedby'], 
        row['createdon'], 
#         row['modifiedon'], 
        row['isActive'], 
        row['CountryID'], 
        row['Province Name (Eng)'], 
        row['Province Name (Thai)']
    )
cursor.execute("SET IDENTITY_INSERT [tl].[PoliticalProvinceMaster] OFF") 

conn.commit()
cursor.close()
conn.close()

## District_insert

In [21]:
dist  = pd.read_csv("District_master_Thai.csv")
dist

,ID,ProvinceID,Province Name (Eng),Province Name (Thai),District Name (Eng),District Name (Thai),isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,1,LOP BURI,ลพบุรี,MUEANG LOP BURI,เมืองลพบุรี,1,NaN,NaN,NaN,NaN,7
1,2,1,LOP BURI,ลพบุรี,PHATTHANA NIKHOM,พัฒนานิคม,1,NaN,NaN,NaN,NaN,7
2,3,1,LOP BURI,ลพบุรี,KHOK SAMRONG,โคกสำโรง,1,NaN,NaN,NaN,NaN,7
3,4,1,LOP BURI,ลพบุรี,CHAI BADAN,ชัยบาดาล,1,NaN,NaN,NaN,NaN,7
4,5,1,LOP BURI,ลพบุรี,THA WUNG,ท่าวุ้ง,1,NaN,NaN,NaN,NaN,7
...,...,...,...,...,...,...,...,...,...,...,...,...
913,914,77,NARATHIWAT,นราธิวาส,SUKHIRIN,สุคิริน,1,NaN,NaN,NaN,NaN,7
914,915,77,NARATHIWAT,นราธิวาส,SU-NGAI KOLOK,สุไหงโก-ลก,1,NaN,NaN,NaN,NaN,7
915,916,77,NARATHIWAT,นราธิวาส,SU-NGAI PADI,สุไหงปาดี,1,NaN,NaN,NaN,NaN,7
916,917,77,NARATHIWAT,นราธิวาส,CHANAE,จะแนะ,1,NaN,NaN,NaN,NaN,7


In [22]:
from datetime import datetime
dist['createdon']= datetime.now()
dist['modifiedon'] = pd.to_datetime(dist['modifiedon'],errors = 'coerce')
dist['createdon'] = pd.to_datetime(dist['createdon'],errors = 'coerce')
dist

,ID,ProvinceID,Province Name (Eng),Province Name (Thai),District Name (Eng),District Name (Thai),isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,1,LOP BURI,ลพบุรี,MUEANG LOP BURI,เมืองลพบุรี,1,2025-09-17 11:01:37.991876,NaT,NaN,NaN,7
1,2,1,LOP BURI,ลพบุรี,PHATTHANA NIKHOM,พัฒนานิคม,1,2025-09-17 11:01:37.991876,NaT,NaN,NaN,7
2,3,1,LOP BURI,ลพบุรี,KHOK SAMRONG,โคกสำโรง,1,2025-09-17 11:01:37.991876,NaT,NaN,NaN,7
3,4,1,LOP BURI,ลพบุรี,CHAI BADAN,ชัยบาดาล,1,2025-09-17 11:01:37.991876,NaT,NaN,NaN,7
4,5,1,LOP BURI,ลพบุรี,THA WUNG,ท่าวุ้ง,1,2025-09-17 11:01:37.991876,NaT,NaN,NaN,7
...,...,...,...,...,...,...,...,...,...,...,...,...
913,914,77,NARATHIWAT,นราธิวาส,SUKHIRIN,สุคิริน,1,2025-09-17 11:01:37.991876,NaT,NaN,NaN,7
914,915,77,NARATHIWAT,นราธิวาส,SU-NGAI KOLOK,สุไหงโก-ลก,1,2025-09-17 11:01:37.991876,NaT,NaN,NaN,7
915,916,77,NARATHIWAT,นราธิวาส,SU-NGAI PADI,สุไหงปาดี,1,2025-09-17 11:01:37.991876,NaT,NaN,NaN,7
916,917,77,NARATHIWAT,นราธิวาส,CHANAE,จะแนะ,1,2025-09-17 11:01:37.991876,NaT,NaN,NaN,7


In [23]:
dist['createdby'] = 'Admin'
dist.isna().sum()

ID                        0
ProvinceID                0
Province Name (Eng)       0
Province Name (Thai)      0
District Name (Eng)       0
District Name (Thai)      0
isActive                  0
createdon                 0
modifiedon              918
createdby                 0
modifiedby              918
CountryID                 0
dtype: int64

In [24]:
dist.columns

Index(['ID', 'ProvinceID', 'Province Name (Eng)', 'Province Name (Thai)',
       'District Name (Eng)', 'District Name (Thai)', 'isActive', 'createdon',
       'modifiedon', 'createdby', 'modifiedby', 'CountryID'],
      dtype='object')

In [28]:
# modifiedOn,
# modifiedBy,
data = [
    (
        row['ID'],                    # politicalDistrictId
        row['ProvinceID'],            # provinceId
        row['createdby'],             # createdBy
#         row['modifiedby'],            # modifiedBy
        row['createdon'],             # createdOn
#         row['modifiedon'],            # modifiedOn
        row['isActive'],              # isActive
        row['CountryID'],             # countryId
        row['District Name (Eng)'],   # districtName
        row['District Name (Thai)']   # districtNameInNative
    )
    for _, row in dist.iterrows()
]

# --- SQL Insert with executemany ---
cursor.execute("SET IDENTITY_INSERT [tl].[PoliticalDistrictMaster] ON")

cursor.executemany("""
    INSERT INTO [tl].[PoliticalDistrictMaster] (
        politicalDistrictId,
        provinceId,
        createdBy,
        
        createdOn,
        
        isActive,
        countryId,
        districtName,
        districtNameInNative
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", data)

cursor.execute("SET IDENTITY_INSERT [tl].[PoliticalDistrictMaster] OFF")

conn.commit()
cursor.close()
conn.close()

## SUb- district_master insert

In [29]:
sub_dist =  pd.read_csv("Sub-District_Master_Thai.csv")
sub_dist

,ID,districtID,District Name (Eng),districtID.1,District Name (Thai),Sub-Dist. (Eng),Sub-Dist. (Thai),isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,1,MUEANG LOP BURI,1,เมืองลพบุรี,THALE CHUP SON,ทะเลชุบศร,1,NaN,NaN,NaN,NaN,7
1,2,1,MUEANG LOP BURI,1,เมืองลพบุรี,THA HIN,ท่าหิน,1,NaN,NaN,NaN,NaN,7
2,3,1,MUEANG LOP BURI,1,เมืองลพบุรี,KOK KO,กกโก,1,NaN,NaN,NaN,NaN,7
3,4,1,MUEANG LOP BURI,1,เมืองลพบุรี,KONG THANU,โก่งธนู,1,NaN,NaN,NaN,NaN,7
4,5,1,MUEANG LOP BURI,1,เมืองลพบุรี,KHAO PHRA NGAM,เขาพระงาม,1,NaN,NaN,NaN,NaN,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5645,5646,917,CHANAE,917,จะแนะ,DU SONG YO,NaN,1,NaN,NaN,NaN,NaN,7
5646,5647,917,CHANAE,917,จะแนะ,PHADUNG MAT,ผดุงมาตร,1,NaN,NaN,NaN,NaN,7
5647,5648,918,CHO-AIRONG,918,เจาะไอร้อง,CHUAP,จวบ,1,NaN,NaN,NaN,NaN,7
5648,5649,918,CHO-AIRONG,918,เจาะไอร้อง,BU KIT,NaN,1,NaN,NaN,NaN,NaN,7


In [30]:
from datetime import datetime
sub_dist['createdon']= datetime.now()
sub_dist['modifiedon'] = pd.to_datetime(sub_dist['modifiedon'],errors = 'coerce')
sub_dist['createdon'] = pd.to_datetime(sub_dist['createdon'],errors = 'coerce')
sub_dist

,ID,districtID,District Name (Eng),districtID.1,District Name (Thai),Sub-Dist. (Eng),Sub-Dist. (Thai),isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,1,MUEANG LOP BURI,1,เมืองลพบุรี,THALE CHUP SON,ทะเลชุบศร,1,2025-09-17 11:11:34.850571,NaT,NaN,NaN,7
1,2,1,MUEANG LOP BURI,1,เมืองลพบุรี,THA HIN,ท่าหิน,1,2025-09-17 11:11:34.850571,NaT,NaN,NaN,7
2,3,1,MUEANG LOP BURI,1,เมืองลพบุรี,KOK KO,กกโก,1,2025-09-17 11:11:34.850571,NaT,NaN,NaN,7
3,4,1,MUEANG LOP BURI,1,เมืองลพบุรี,KONG THANU,โก่งธนู,1,2025-09-17 11:11:34.850571,NaT,NaN,NaN,7
4,5,1,MUEANG LOP BURI,1,เมืองลพบุรี,KHAO PHRA NGAM,เขาพระงาม,1,2025-09-17 11:11:34.850571,NaT,NaN,NaN,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5645,5646,917,CHANAE,917,จะแนะ,DU SONG YO,NaN,1,2025-09-17 11:11:34.850571,NaT,NaN,NaN,7
5646,5647,917,CHANAE,917,จะแนะ,PHADUNG MAT,ผดุงมาตร,1,2025-09-17 11:11:34.850571,NaT,NaN,NaN,7
5647,5648,918,CHO-AIRONG,918,เจาะไอร้อง,CHUAP,จวบ,1,2025-09-17 11:11:34.850571,NaT,NaN,NaN,7
5648,5649,918,CHO-AIRONG,918,เจาะไอร้อง,BU KIT,NaN,1,2025-09-17 11:11:34.850571,NaT,NaN,NaN,7


In [31]:
sub_dist['createdby'] = 'Admin'
sub_dist.isna().sum()

ID                         0
districtID                 0
District Name (Eng)        0
districtID.1               0
District Name (Thai)       0
Sub-Dist. (Eng)            0
Sub-Dist. (Thai)          58
isActive                   0
createdon                  0
modifiedon              5650
createdby                  0
modifiedby              5650
CountryID                  0
dtype: int64

In [32]:
sub_dist.columns

Index(['ID', 'districtID', 'District Name (Eng)', 'districtID.1',
       'District Name (Thai)', 'Sub-Dist. (Eng)', 'Sub-Dist. (Thai)',
       'isActive', 'createdon', 'modifiedon', 'createdby', 'modifiedby',
       'CountryID'],
      dtype='object')

In [33]:
sub_dist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5650 entries, 0 to 5649
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   ID                    5650 non-null   int64         
 1   districtID            5650 non-null   int64         
 2   District Name (Eng)   5650 non-null   object        
 3   districtID.1          5650 non-null   int64         
 4   District Name (Thai)  5650 non-null   object        
 5   Sub-Dist. (Eng)       5650 non-null   object        
 6   Sub-Dist. (Thai)      5592 non-null   object        
 7   isActive              5650 non-null   int64         
 8   createdon             5650 non-null   datetime64[us]
 9   modifiedon            0 non-null      datetime64[ns]
 10  createdby             5650 non-null   object        
 11  modifiedby            0 non-null      float64       
 12  CountryID             5650 non-null   int64         
dtypes: datetime64[ns](

In [57]:
import pyodbc

conn = pyodbc.connect(
    'DRIVER={ODBC Driver 18 for SQL Server};'
    'SERVER=phazd1856sqlserver.database.windows.net;'
    'DATABASE=devsqldatabase;'
    'UID=administratorLogin;'
    'PWD=Admin@8874;'
    'Encrypt=yes;'
    'TrustServerCertificate=no;'
    'Connection Timeout=30;'
)
print(" Connected to Azure SQL Server successfully!")

cursor = conn.cursor()

 Connected to Azure SQL Server successfully!


In [44]:
import pandas as pd
import pyodbc

# ---- Step 1: Prepare data ----
data = [
    (
        int(row['ID']),                               # politicalSubDistrictId
        int(row['districtID']) if pd.notna(row['districtID']) else None,   # districtId
        None,                                         # sfdcId (not in df)
        str(row.get('createdby')) if pd.notna(row.get('createdby')) else None,   # createdBy
        str(row.get('modifiedby')) if pd.notna(row.get('modifiedby')) else None, # modifiedBy
        row['createdon'].to_pydatetime() if pd.notna(row.get('createdon')) else None,  # createdOn
        row['modifiedon'].to_pydatetime() if pd.notna(row.get('modifiedon')) else None, # modifiedOn
        int(row['isActive']) if pd.notna(row['isActive']) else 1,            # isActive
        int(row['CountryID']) if pd.notna(row['CountryID']) else None,       # countryId
        str(row['Sub-Dist. (Eng)']) if pd.notna(row['Sub-Dist. (Eng)']) else None,   # subDistrictName
        str(row['Sub-Dist. (Thai)']) if pd.notna(row['Sub-Dist. (Thai)']) else None   # subDistrictNameInNative
    )
    for _, row in sub_dist.iterrows()
]

# ---- Step 2: SQL Insert ----
sql = """
INSERT INTO [tl].[PoliticalSubDistrictMaster] (
    politicalSubDistrictId,
    districtId,
    sfdcId,
    createdBy,
    modifiedBy,
    createdOn,
    modifiedOn,
    isActive,
    countryId,
    subDistrictName,
    subDistrictNameInNative
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

# ---- Step 3: Execute with fast_executemany ----
# cursor = conn.cursor()
cursor.fast_executemany = True

cursor.execute("SET IDENTITY_INSERT [tl].[PoliticalSubDistrictMaster] ON")
cursor.executemany(sql, data)
cursor.execute("SET IDENTITY_INSERT [tl].[PoliticalSubDistrictMaster] OFF")

conn.commit()
cursor.close()
conn.close()


ProgrammingError: Attempt to use a closed cursor.

In [ ]:
# this is done 

## Region_master_insert

In [46]:
reg_mas = pd.read_csv("Region_master_Thai.csv")
reg_mas

,ID,Region,BUID,isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,LOWER NORTH,1,1,NaN,NaN,NaN,NaN,7
1,2,CENTRAL,1,1,NaN,NaN,NaN,NaN,7
2,3,NORTH EAST,1,1,NaN,NaN,NaN,NaN,7
3,4,LOWER NORTH,2,1,NaN,NaN,NaN,NaN,7
4,5,NORTH EAST,2,1,NaN,NaN,NaN,NaN,7
5,6,UPPER NORTH,2,1,NaN,NaN,NaN,NaN,7


In [47]:
from datetime import datetime
reg_mas['createdon']= datetime.now()
reg_mas['modifiedon'] = pd.to_datetime(reg_mas['modifiedon'],errors = 'coerce')
reg_mas['createdon'] = pd.to_datetime(reg_mas['createdon'],errors = 'coerce')
reg_mas

,ID,Region,BUID,isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,LOWER NORTH,1,1,2025-09-17 11:41:04.442555,NaT,NaN,NaN,7
1,2,CENTRAL,1,1,2025-09-17 11:41:04.442555,NaT,NaN,NaN,7
2,3,NORTH EAST,1,1,2025-09-17 11:41:04.442555,NaT,NaN,NaN,7
3,4,LOWER NORTH,2,1,2025-09-17 11:41:04.442555,NaT,NaN,NaN,7
4,5,NORTH EAST,2,1,2025-09-17 11:41:04.442555,NaT,NaN,NaN,7
5,6,UPPER NORTH,2,1,2025-09-17 11:41:04.442555,NaT,NaN,NaN,7


In [48]:
reg_mas['createdby'] = 'Admin'
reg_mas.isna().sum()

ID            0
Region        0
BUID          0
isActive      0
createdon     0
modifiedon    6
createdby     0
modifiedby    6
CountryID     0
dtype: int64

In [49]:
reg_mas.columns

Index(['ID', 'Region', 'BUID', 'isActive', 'createdon', 'modifiedon',
       'createdby', 'modifiedby', 'CountryID'],
      dtype='object')

In [51]:
# regionNameInNative,
# modifiedBy,
# modifiedOn
#  sfdcId,

data = [
    (
        int(row['ID']),                             # CortevaRegionId
#         None,                                       # sfdcId (not in df)
        str(row.get('createdby')) if pd.notna(row.get('createdby')) else None,   # createdBy
#         str(row.get('modifiedby')) if pd.notna(row.get('modifiedby')) else None, # modifiedBy
        row['createdon'].to_pydatetime() if pd.notna(row.get('createdon')) else None,  # createdOn
#         row['modifiedon'].to_pydatetime() if pd.notna(row.get('modifiedon')) else None, # modifiedOn
        int(row['isActive']) if pd.notna(row['isActive']) else 1,               # isActive
        int(row['CountryID']) if pd.notna(row['CountryID']) else None,          # countryId
        str(row['Region']) if pd.notna(row['Region']) else None,                # regionName
#         None,                                       # regionNameInNative (not in df)
        int(row['BUID']) if pd.notna(row['BUID']) else None                     # businessUnitId
    )
    for _, row in reg_mas.iterrows()
]

# ---- Step 2: SQL Insert ----
sql = """
INSERT INTO [tl].[CortevaRegionMaster] (
    CortevaRegionId,
   
    createdBy,
    
    createdOn,
    
    isActive,
    countryId,
    regionName,
    
    businessUnitId
)
VALUES (?, ?, ?, ?, ?, ?, ?)
"""

# ---- Step 3: Execute with fast_executemany ----
# cursor = conn.cursor()
cursor.fast_executemany = True

cursor.execute("SET IDENTITY_INSERT [tl].[CortevaRegionMaster] ON")
cursor.executemany(sql, data)
cursor.execute("SET IDENTITY_INSERT [tl].[CortevaRegionMaster] OFF")

conn.commit()
cursor.close()
conn.close()


## territory_master_insert

In [52]:
ter_mas = pd.read_csv("Territory_Master_Thai.csv")
ter_mas

,ID,RegionID,BUID,Territory,isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,1,1,LN2,1,NaN,NaN,NaN,NaN,7
1,2,1,1,LN1,1,NaN,NaN,NaN,NaN,7
2,3,2,1,C2,1,NaN,NaN,NaN,NaN,7
3,4,2,1,C1,1,NaN,NaN,NaN,NaN,7
4,5,3,1,NE5,1,NaN,NaN,NaN,NaN,7
5,6,4,2,LN3,1,NaN,NaN,NaN,NaN,7
6,7,5,2,NE1,1,NaN,NaN,NaN,NaN,7
7,8,5,2,NE2,1,NaN,NaN,NaN,NaN,7
8,9,5,2,NE3,1,NaN,NaN,NaN,NaN,7
9,10,5,2,NE4,1,NaN,NaN,NaN,NaN,7


In [53]:
from datetime import datetime
ter_mas['createdon']= datetime.now()
ter_mas['modifiedon'] = pd.to_datetime(ter_mas['modifiedon'],errors = 'coerce')
ter_mas['createdon'] = pd.to_datetime(ter_mas['createdon'],errors = 'coerce')
ter_mas

,ID,RegionID,BUID,Territory,isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,1,1,LN2,1,2025-09-17 11:53:16.224960,NaT,NaN,NaN,7
1,2,1,1,LN1,1,2025-09-17 11:53:16.224960,NaT,NaN,NaN,7
2,3,2,1,C2,1,2025-09-17 11:53:16.224960,NaT,NaN,NaN,7
3,4,2,1,C1,1,2025-09-17 11:53:16.224960,NaT,NaN,NaN,7
4,5,3,1,NE5,1,2025-09-17 11:53:16.224960,NaT,NaN,NaN,7
5,6,4,2,LN3,1,2025-09-17 11:53:16.224960,NaT,NaN,NaN,7
6,7,5,2,NE1,1,2025-09-17 11:53:16.224960,NaT,NaN,NaN,7
7,8,5,2,NE2,1,2025-09-17 11:53:16.224960,NaT,NaN,NaN,7
8,9,5,2,NE3,1,2025-09-17 11:53:16.224960,NaT,NaN,NaN,7
9,10,5,2,NE4,1,2025-09-17 11:53:16.224960,NaT,NaN,NaN,7


In [54]:
ter_mas['createdby'] = 'Admin'
ter_mas.isna().sum()

ID             0
RegionID       0
BUID           0
Territory      0
isActive       0
createdon      0
modifiedon    13
createdby      0
modifiedby    13
CountryID      0
dtype: int64

In [55]:
ter_mas.columns

Index(['ID', 'RegionID', 'BUID', 'Territory', 'isActive', 'createdon',
       'modifiedon', 'createdby', 'modifiedby', 'CountryID'],
      dtype='object')

In [56]:
ter_mas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   ID          13 non-null     int64         
 1   RegionID    13 non-null     int64         
 2   BUID        13 non-null     int64         
 3   Territory   13 non-null     object        
 4   isActive    13 non-null     int64         
 5   createdon   13 non-null     datetime64[us]
 6   modifiedon  0 non-null      datetime64[ns]
 7   createdby   13 non-null     object        
 8   modifiedby  0 non-null      float64       
 9   CountryID   13 non-null     int64         
dtypes: datetime64[ns](1), datetime64[us](1), float64(1), int64(5), object(2)
memory usage: 1.1+ KB


In [58]:
# sfdcId,
# modifiedBy,
# modifiedOn,
# territoryNameInNative,

cursor.fast_executemany = True

data = [
    (
        int(row['ID']),                             # cortevaTerritoryId
        int(row['RegionID']) if pd.notna(row['RegionID']) else None,   # regionId
#         None,                                       # sfdcId (not in df)
        str(row.get('createdby')) if pd.notna(row.get('createdby')) else None,   # createdBy
#         str(row.get('modifiedby')) if pd.notna(row.get('modifiedby')) else None, # modifiedBy
        row['createdon'].to_pydatetime() if pd.notna(row['createdon']) else None,  # createdOn
#         row['modifiedon'].to_pydatetime() if pd.notna(row['modifiedon']) else None, # modifiedOn
        int(row['isActive']) if pd.notna(row['isActive']) else 1,       # isActive
        int(row['CountryID']) if pd.notna(row['CountryID']) else None,  # countryId
        str(row['Territory']) if pd.notna(row['Territory']) else None,  # territoryName
#         None,                                       # territoryNameInNative (not in df)
        int(row['BUID']) if pd.notna(row['BUID']) else None             # businessUnitId
    )
    for _, row in ter_mas.iterrows()
]

sql = """
INSERT INTO [tl].[CortevaTerritoryMaster] (
    cortevaTerritoryId,
    regionId,
    
    createdBy,
    
    createdOn,
    
    isActive,
    countryId,
    territoryName,
    
    businessUnitId
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
"""

cursor.execute("SET IDENTITY_INSERT [tl].[CortevaTerritoryMaster] ON")
cursor.executemany(sql, data)
cursor.execute("SET IDENTITY_INSERT [tl].[CortevaTerritoryMaster] OFF")

conn.commit()
cursor.close()
conn.close()


## sub_ter_mast_insert

In [62]:
sub_ter_mas = pd.read_csv("SUB_Territory_Master_Thai.csv")
sub_ter_mas.shape

(43, 10)

In [63]:
from datetime import datetime
sub_ter_mas['createdon']= datetime.now()
sub_ter_mas['modifiedon'] = pd.to_datetime(sub_ter_mas['modifiedon'],errors = 'coerce')
sub_ter_mas['createdon'] = pd.to_datetime(sub_ter_mas['createdon'],errors = 'coerce')
sub_ter_mas.head()

,ID,TerrtoryID,BUID,Sub-Territory,isActive,createdon,modifiedon,createdby,modifiedby,CountryID
0,1,1,1,LN2 B,1,2025-09-17 12:02:01.743583,NaT,NaN,NaN,7
1,2,1,1,LN2 D,1,2025-09-17 12:02:01.743583,NaT,NaN,NaN,7
2,3,1,1,LN2 C,1,2025-09-17 12:02:01.743583,NaT,NaN,NaN,7
3,4,1,1,LN2 A,1,2025-09-17 12:02:01.743583,NaT,NaN,NaN,7
4,5,1,1,LN2 E,1,2025-09-17 12:02:01.743583,NaT,NaN,NaN,7


In [64]:
sub_ter_mas['createdby'] = 'Admin'
sub_ter_mas.isna().sum()

ID                0
TerrtoryID        0
BUID              0
Sub-Territory     0
isActive          0
createdon         0
modifiedon       43
createdby         0
modifiedby       43
CountryID         0
dtype: int64

In [65]:
sub_ter_mas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   ID             43 non-null     int64         
 1   TerrtoryID     43 non-null     int64         
 2   BUID           43 non-null     int64         
 3   Sub-Territory  43 non-null     object        
 4   isActive       43 non-null     int64         
 5   createdon      43 non-null     datetime64[us]
 6   modifiedon     0 non-null      datetime64[ns]
 7   createdby      43 non-null     object        
 8   modifiedby     0 non-null      float64       
 9   CountryID      43 non-null     int64         
dtypes: datetime64[ns](1), datetime64[us](1), float64(1), int64(5), object(2)
memory usage: 3.5+ KB


In [66]:
sub_ter_mas.columns

Index(['ID', 'TerrtoryID', 'BUID', 'Sub-Territory', 'isActive', 'createdon',
       'modifiedon', 'createdby', 'modifiedby', 'CountryID'],
      dtype='object')

In [72]:
import pyodbc

conn = pyodbc.connect(
    'DRIVER={ODBC Driver 18 for SQL Server};'
    'SERVER=phazd1856sqlserver.database.windows.net;'
    'DATABASE=devsqldatabase;'
    'UID=administratorLogin;'
    'PWD=Admin@8874;'
    'Encrypt=yes;'
    'TrustServerCertificate=no;'
    'Connection Timeout=30;'
)
print(" Connected to Azure SQL Server successfully!")

cursor = conn.cursor()

 Connected to Azure SQL Server successfully!


In [73]:
# sfdcId,
# modifiedBy,
# modifiedOn,
# subTerritoryNameInNative,

cursor.fast_executemany = True

data = [
    (
        int(row['ID']),                             # cortevaSubTerritoryId
        int(row['TerrtoryID']) if pd.notna(row['TerrtoryID']) else None,  # territoryId
#         None,                                       # sfdcId (not in df)
        str(row.get('createdby')) if pd.notna(row.get('createdby')) else None,   # createdBy
#         str(row.get('modifiedby')) if pd.notna(row.get('modifiedby')) else None, # modifiedBy
        row['createdon'].to_pydatetime() if pd.notna(row['createdon']) else None,  # createdOn
#         row['modifiedon'].to_pydatetime() if pd.notna(row['modifiedon']) else None, # modifiedOn
        int(row['isActive']) if pd.notna(row['isActive']) else 1,       # isActive
        int(row['CountryID']) if pd.notna(row['CountryID']) else None,  # countryId
        str(row['Sub-Territory']) if pd.notna(row['Sub-Territory']) else None,  # subTerritoryName
#         None,                                       # subTerritoryNameInNative (not in df)
        int(row['BUID']) if pd.notna(row['BUID']) else None             # businessUnitId
    )
    for _, row in sub_ter_mas.iterrows()
]

sql = """
INSERT INTO [tl].[CortevaSubTerritoryMaster] (
    cortevaSubTerritoryId,
    territoryId,
    
    createdBy,
    
    createdOn,
    
    isActive,
    countryId,
    subTerritoryName,
    
    businessUnitId
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
"""

cursor.execute("SET IDENTITY_INSERT [tl].[CortevaSubTerritoryMaster] ON")
cursor.executemany(sql, data)
cursor.execute("SET IDENTITY_INSERT [tl].[CortevaSubTerritoryMaster] OFF")

conn.commit()
cursor.close()
conn.close()

## Employee_master_insert

In [ ]:
emp_mas = pd.read_csv()